# PySpark 50 — LeetCode-Style Problem Pack

50 hands-on PySpark exercises sorted by topic and difficulty.
Each cell has a **problem statement**, then a **solution cell** below it.
Try solving before peeking at the solution.

**Setup:** assumes `pyspark` is installed and a `SparkSession` is available.
Run the setup cell first, then work through each problem.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, types as T, Row
import os

spark = SparkSession.builder.appName("pyspark-50").getOrCreate()
DATA = os.path.join(os.getcwd(), "..", "data")

---
## Section 1 — DataFrame Basics (Q1–Q8)
Reading data, inspecting schemas, selecting, filtering, sorting, limiting.

In [ ]:
# Q1: Read orders.csv and print the schema
orders_path = os.path.join(DATA, "orders.csv")
# your code here

In [ ]:
# Solution Q1
df = spark.read.csv(orders_path, header=True, inferSchema=True)
df.printSchema()

In [ ]:
# Q2: Select only order_id, amount, country columns. Show 5 rows.
# your code here

In [ ]:
# Solution Q2
df.select("order_id", "amount", "country").show(5)

In [ ]:
# Q3: Filter orders where amount > 50. How many rows?
# your code here

In [ ]:
# Solution Q3
df.filter(F.col("amount") > 50).count()

In [ ]:
# Q4: Filter DELIVERED orders, sort by amount descending.
# your code here

In [ ]:
# Solution Q4
df.filter(F.col("status") == "DELIVERED").orderBy(F.col("amount").desc()).show()

In [ ]:
# Q5: Read users.json (multiline) and show the flattened structure.
# Hint: use from_json or read with multiLine
# your code here

In [ ]:
# Solution Q5
users_path = os.path.join(DATA, "users.json")
users_df = spark.read.option("multiLine", "true").json(users_path)
users_df.printSchema()
users_df.show(truncate=False)

In [ ]:
# Q6: Count distinct countries in orders.csv.
# your code here

In [ ]:
# Solution Q6
df.select("country").distinct().count()

In [ ]:
# Q7: Read orders_extended.csv and display the first 3 rows as a list of Rows.
# your code here

In [ ]:
# Solution Q7
ext_path = os.path.join(DATA, "orders_extended.csv")
ext = spark.read.csv(ext_path, header=True, inferSchema=True)
ext.take(3)

In [ ]:
# Q8: Read events_sessions.csv and filter only 'click' events.
# your code here

In [ ]:
# Solution Q8
sessions_path = os.path.join(DATA, "events_sessions.csv")
sessions = spark.read.csv(sessions_path, header=True, inferSchema=True)
sessions.filter(F.col("event") == "click").show()

---
## Section 2 — Column & Row Operations (Q9–Q15)
withColumn, when/otherwise, alias, drop, distinct, sample.

In [ ]:
# Q9: Add a column 'is_high_value' that is True when amount > 100.
# your code here

In [ ]:
# Solution Q9
df.withColumn("is_high_value", F.col("amount") > 100).show()

In [ ]:
# Q10: Create a column 'tier_label' using when/otherwise:
#   amount > 100 -> 'premium', amount > 50 -> 'standard', else 'basic'
# your code here

In [ ]:
# Solution Q10
df.withColumn("tier_label",
    F.when(F.col("amount") > 100, "premium")
     .when(F.col("amount") > 50, "standard")
     .otherwise("basic")
).show()

In [ ]:
# Q11: Rename 'amount' to 'order_amount' and 'status' to 'order_status'.
# your code here

In [ ]:
# Solution Q11
df.withColumnRenamed("amount", "order_amount").withColumnRenamed("status", "order_status").show()

In [ ]:
# Q12: Drop the 'status' column and show the result.
# your code here

In [ ]:
# Solution Q12
df.drop("status").show()

In [ ]:
# Q13: Fill null amounts with 0.0 (introduce a null first if needed).
# Create a copy with a null, then fill.
from pyspark.sql import Row
df_with_null = spark.createDataFrame([
    Row(order_id=99, customer_id=999, order_ts="2026-03-01T00:00:00Z", status="PENDING", amount=None, country="UK")
]) \
    .unionByName(df, allowMissingColumns=True)
# your code here

In [ ]:
# Solution Q13
df_with_null.fillna({"amount": 0.0}).show()

In [ ]:
# Q14: Remove duplicate rows based on (customer_id, status).
# your code here

In [ ]:
# Solution Q14
df.dropDuplicates(["customer_id", "status"]).show()

In [ ]:
# Q15: Sample 50% of the orders table without replacement (seed=42).
# your code here

In [ ]:
# Solution Q15
df.sample(withReplacement=False, fraction=0.5, seed=42).show()

---
## Section 3 — Aggregations & GroupBy (Q16–Q23)
groupBy, agg, countDistinct, sum, avg, pivot, cube, rollup.

In [ ]:
# Q16: Count total orders per status.
# your code here

In [ ]:
# Solution Q16
df.groupBy("status").count().show()

In [ ]:
# Q17: Sum of amount per country, sorted by total descending.
# your code here

In [ ]:
# Solution Q17
df.groupBy("country").agg(F.sum("amount").alias("total_amount")).orderBy(F.col("total_amount").desc()).show()

In [ ]:
# Q18: Per status, compute: count, avg amount, max amount, min amount.
# your code here

In [ ]:
# Solution Q18
df.groupBy("status").agg(
    F.count("*").alias("cnt"),
    F.avg("amount").alias("avg_amount"),
    F.max("amount").alias("max_amount"),
    F.min("amount").alias("min_amount")
).show()

In [ ]:
# Q19: Count distinct customers per country.
# your code here

In [ ]:
# Solution Q19
df.groupBy("country").agg(F.countDistinct("customer_id").alias("unique_customers")).show()

In [ ]:
# Q20: Pivot the orders table — status as columns, sum of amount per country.
# your code here

In [ ]:
# Solution Q20
df.groupBy("country").pivot("status").agg(F.sum("amount")).show()

In [ ]:
# Q21: Use cube on (country, status) and show sum of amounts.
# your code here

In [ ]:
# Solution Q21
df.cube("country", "status").agg(F.sum("amount").alias("total")).orderBy("country", "status").show()

In [ ]:
# Q22: Collect list of order_ids per status using collect_list.
# your code here

In [ ]:
# Solution Q22
df.groupBy("status").agg(F.collect_list("order_id").alias("order_ids")).show()

In [ ]:
# Q23: Using orders_extended, find the total revenue per product.
# your code here

In [ ]:
# Solution Q23
ext.groupBy("product").agg(F.sum("amount").alias("revenue")).orderBy(F.col("revenue").desc()).show()

---
## Section 4 — Joins (Q24–Q30)
Inner, left, right, anti, semi, broadcast hint, self-joins.

In [ ]:
# Q24: Inner-join orders with users on customer_id = user.id.
# Show order_id, customer_id, name, tier, amount.
# your code here

In [ ]:
# Solution Q24
users_flat = users_df.select(
    F.col("user.id").alias("user_id"),
    F.col("user.name"),
    F.col("user.tier")
)
joined = df.join(users_flat, df.customer_id == users_flat.user_id, "inner")
joined.select("order_id", "customer_id", "name", "tier", "amount").show()

In [ ]:
# Q25: Left join — all orders, with user name if available.
# your code here

In [ ]:
# Solution Q25
df.join(users_flat, df.customer_id == users_flat.user_id, "left").select(
    "order_id", "customer_id", "name", "amount"
).show()

In [ ]:
# Q26: Anti-join — customers who have NO orders.
# your code here

In [ ]:
# Solution Q26
users_flat.join(df, users_flat.user_id == df.customer_id, "anti").show()

In [ ]:
# Q27: Semi-join — users who have placed at least one order.
# your code here

In [ ]:
# Solution Q27
users_flat.join(df, users_flat.user_id == df.customer_id, "semi").show()

In [ ]:
# Q28: Self-join on orders_extended — find orders from the same customer
# where the 2nd order happened after the 1st.
# Show: customer_id, first_order_id, second_order_id, first_ts, second_ts.
# your code here

In [ ]:
# Solution Q28
a = ext.alias("a")
b = ext.alias("b")
self_joined = a.join(
    b,
    (F.col("a.customer_id") == F.col("b.customer_id")) &
    (F.col("a.order_ts") < F.col("b.order_ts")),
    "inner"
)
self_joined.select(
    F.col("a.customer_id"),
    F.col("a.order_id").alias("first_order_id"),
    F.col("b.order_id").alias("second_order_id"),
    F.col("a.order_ts").alias("first_ts"),
    F.col("b.order_ts").alias("second_ts")
).show()

In [ ]:
# Q29: Use broadcast hint to join a small DataFrame (users) with orders.
# your code here

In [ ]:
# Solution Q29
from pyspark.sql.functions import broadcast
df.join(broadcast(users_flat), df.customer_id == users_flat.user_id, "inner").show()

In [ ]:
# Q30: Full outer join orders and users. Count rows before and after.
# your code here

In [ ]:
# Solution Q30
print("orders:", df.count(), "users:", users_flat.count())
full = df.join(users_flat, df.customer_id == users_flat.user_id, "outer")
print("full outer:", full.count())
full.show()

---
## Section 5 — Window Functions (Q31–Q38)
ROW_NUMBER, RANK, DENSE_RANK, LAG, LEAD, running totals, NTILE.

In [ ]:
# Q31: Rank orders by amount descending per country.
# Show order_id, country, amount, rank.
# your code here

In [ ]:
# Solution Q31
from pyspark.sql.window import Window
w = Window.partitionBy("country").orderBy(F.col("amount").desc())
df.withColumn("rank", F.rank().over(w)).show()

In [ ]:
# Q32: Use DENSE_RANK to rank orders per country by amount.
# Show the difference from Q31 when there are ties.
# your code here

In [ ]:
# Solution Q32
df.withColumn("dense_rank", F.dense_rank().over(w)).show()

In [ ]:
# Q33: Per customer, find the previous order amount using LAG.
# Order by order_ts ascending.
# your code here

In [ ]:
# Solution Q33
w2 = Window.partitionBy("customer_id").orderBy("order_ts")
ext.withColumn("prev_amount", F.lag("amount").over(w2)).select(
    "customer_id", "order_id", "amount", "prev_amount"
).show()

In [ ]:
# Q34: Per customer, find the next order amount using LEAD.
# your code here

In [ ]:
# Solution Q34
ext.withColumn("next_amount", F.lead("amount").over(w2)).select(
    "customer_id", "order_id", "amount", "next_amount"
).show()

In [ ]:
# Q35: Running total of amount per customer, ordered by order_ts.
# your code here

In [ ]:
# Solution Q35
ext.withColumn("running_total", F.sum("amount").over(w2.rowsBetween(Window.unboundedPreceding, 0))).show()

In [ ]:
# Q36: NTILE(4) — divide orders into 4 quartiles per country by amount.
# your code here

In [ ]:
# Solution Q36
df.withColumn("quartile", F.ntile(4).over(w)).show()

In [ ]:
# Q37: Per country, find the order with the 2nd highest amount.
# your code here

In [ ]:
# Solution Q37
ranked = df.withColumn("rn", F.row_number().over(w))
ranked.filter(F.col("rn") == 2).drop("rn").show()

In [ ]:
# Q38: Count cumulative orders per country over time (ordered by order_ts).
# Show: order_ts, country, cum_count.
# your code here

In [ ]:
# Solution Q38
w3 = Window.partitionBy("country").orderBy("order_ts").rowsBetween(Window.unboundedPreceding, 0)
df.withColumn("cum_count", F.count("*").over(w3)).select("order_ts", "country", "cum_count").show()

---
## Section 6 — Complex Types (Q39–Q44)
Arrays, structs, JSON parsing, explode, split, map keys.

In [ ]:
# Q39: Parse the nested JSON in users.json to extract user.id, user.name, user.tier,
# geo.country, geo.city into top-level columns.
# your code here

In [ ]:
# Solution Q39
users_df.select(
    F.col("user.id").alias("user_id"),
    F.col("user.name"),
    F.col("user.tier"),
    F.col("geo.country"),
    F.col("geo.city")
).show()

In [ ]:
# Q40: Read payloads.csv. Parse the 'payload' JSON string column and extract
# order_id, items array, and source from meta.
# Hint: use from_json with a defined schema.
# your code here

In [ ]:
# Solution Q40
payload_path = os.path.join(DATA, "payloads.csv")
payloads = spark.read.csv(payload_path, header=True, inferSchema=True)

item_schema = T.StructType([
    T.StructField("sku", T.StringType()),
    T.StructField("qty", T.IntegerType())
])
payload_schema = T.StructType([
    T.StructField("order_id", T.IntegerType()),
    T.StructField("items", T.ArrayType(item_schema)),
    T.StructField("meta", T.StructType([
        T.StructField("source", T.StringType())
    ]))
])

parsed = payloads.withColumn("parsed", F.from_json(F.col("payload"), payload_schema))
parsed.select(
    "id",
    F.col("parsed.order_id").alias("parsed_order_id"),
    F.col("parsed.items"),
    F.col("parsed.meta.source").alias("source")
).show(truncate=False)

In [ ]:
# Q41: From the parsed payloads above, explode the items array so each item becomes a row.
# your code here

In [ ]:
# Solution Q41
parsed.select("id", F.explode(F.col("parsed.items")).alias("item")).select(
    "id",
    F.col("item.sku"),
    F.col("item.qty")
).show()

In [ ]:
# Q42: Create an array column from order_id and amount, then compute its size and sum.
# your code here

In [ ]:
# Solution Q42
arr_df = df.withColumn("arr", F.array("order_id", F.col("amount").cast("int")))
arr_df.select(
    "order_id",
    "arr",
    F.size("arr").alias("arr_size"),
    F.expr("aggregate(arr, 0, (acc, x) -> acc + x)").alias("arr_sum")
).show()

In [ ]:
# Q43: Split the 'event' column from events_sessions into characters and compute the length.
# your code here

In [ ]:
# Solution Q43
sessions.withColumn("chars", F.split(F.col("event"), ""))\
    .withColumn("num_chars", F.size("chars"))\
    .show()

In [ ]:
# Q44: Convert each row in orders to a struct, then query fields from the struct.
# your code here

In [ ]:
# Solution Q44
struct_df = df.select(F.struct("order_id", "amount", "status").alias("order_struct"))
struct_df.select(
    F.col("order_struct.order_id"),
    F.col("order_struct.amount"),
    F.col("order_struct.status")
).show()

---
## Section 7 — Optimization & Advanced (Q45–Q50)
Repartition, coalesce, caching, UDFs, SQL temp views, explain plans.

In [ ]:
# Q45: Repartition the orders DataFrame to 4 partitions. Check the partition count.
# your code here

In [ ]:
# Solution Q45
repartitioned = df.repartition(4)
print("Partitions:", repartitioned.rdd.getNumPartitions())

In [ ]:
# Q46: Coalesce the DataFrame to 2 partitions. Why might you prefer coalesce over repartition?
# your code here

In [ ]:
# Solution Q46
coalesced = df.coalesce(2)
print("Partitions:", coalesced.rdd.getNumPartitions())
# Coalesce avoids a full shuffle (only merges existing partitions).
# Repartition does a full shuffle and can increase or decrease partitions.

In [ ]:
# Q47: Cache the orders DataFrame, trigger an action, then uncache.
# Verify it's cached with spark.catalog.
# your code here

In [ ]:
# Solution Q47
df.cache()
df.count()  # materialise cache
print("Is cached:", spark.catalog.isCached("orders"))
df.unpersist()

In [ ]:
# Q48: Register orders as a temp view and run a SQL query:
# "SELECT status, SUM(amount) as total FROM orders GROUP BY status ORDER BY total DESC"
# your code here

In [ ]:
# Solution Q48
df.createOrReplaceTempView("orders")
spark.sql("""
    SELECT status, SUM(amount) as total
    FROM orders
    GROUP BY status
    ORDER BY total DESC
""").show()

In [ ]:
# Q49: Create a Python UDF that categorises amount into 'Low', 'Medium', 'High'.
# Register and apply it.
# your code here

In [ ]:
# Solution Q49
@F.udf(returnType=T.StringType())
def amount_category(amt):
    if amt is None:
        return "Unknown"
    if amt > 100:
        return "High"
    elif amt > 50:
        return "Medium"
    else:
        return "Low"

df.withColumn("category", amount_category(F.col("amount"))).show()

In [ ]:
# Q50: Show the physical plan (explain) for a filtered + grouped query.
# Then use .explain("cost") or .explain("formatted") for more detail.
# your code here

In [ ]:
# Solution Q50
query = df.filter(F.col("country") == "IN").groupBy("status").agg(F.sum("amount").alias("total"))
query.explain("formatted")

---
## Bonus — Quick Cheat Sheet

| Pattern | Code |
|---|---|
| Read CSV | `spark.read.csv(path, header=True, inferSchema=True)` |
| Read JSON | `spark.read.json(path)` |
| Filter | `df.filter(F.col('x') > 5)` |
| Add column | `df.withColumn('new', F.col('a') * 2)` |
| Group + agg | `df.groupBy('k').agg(F.sum('v').alias('total'))` |
| Window rank | `F.rank().over(Window.partitionBy('k').orderBy('v'))` |
| Join | `a.join(b, a.k == b.k, 'inner')` |
| Broadcast join | `a.join(broadcast(b), a.k == b.k)` |
| SQL | `df.createOrReplaceTempView('t'); spark.sql('SELECT * FROM t')` |
| UDF | `@F.udf(returnType=T.StringType()); def f(x): ...` |
| Explode | `df.withColumn('item', F.explode('items'))` |
| Cache | `df.cache(); df.count(); df.unpersist()` |
| Repartition | `df.repartition(n)` or `df.coalesce(n)` |
| Explain | `df.explain('formatted')` |